## Mini ML Competition 수행을 위한 template 파일

1. https://agtechresearch.pythonanywhere.com/ 에 접속하여 회원가입해 주세요. (비밀번호는 단순하게 만드는 것을 권장. 예: 1234)
2. `username` 에 이메일 형식의 아이디를 기입해 주세요.
3. `password` 에 비밀번호를 기입해 주세요.

In [ ]:
project = "globalbeans"  # 수정하지 마세요
username = ""  # 회원가입 시 사용한 이메일아이디 (예시. abc@hello.com)
password = ""  # 비밀번호

리더보드 제출을 위한 기본 설정: 아래 코드를 실행해주세요.


In [ ]:
import os
import urllib.request

if not os.path.exists("competition.py"):
    url = "https://raw.githubusercontent.com/agtechresearch/MLapplications-graduate/refs/heads/main/competition/competition.py"
    filename = "competition.py"
    urllib.request.urlretrieve(url, filename)

아래 코드를 실행하여 데이터를 다운로드 받습니다: 3개의 csv 파일이 data 폴더에 다운로드됨

 * dataset.csv: 과거 품종관련 데이터 -> 학습에 사용할 데이터셋
 * problem.csv: 현재 업체가 수입/판매를 진행하고자 하는 400건의 강낭콩 정보 -> ML 모델에 의하여 품종 예측을 수행하여야 할 데이터셋
 * submission.csv: 리더보드 서버 제출을 위한 파일 형식


In [ ]:
import competition

# 파일 다운로드
competition.download_competition_files(project)

----------

### 아래는 랜덤포레스트를 사용하여 강낭콩 품종 예측 모델을 만들고, 코랩환경에서 결과를 리더보드에 제출하는 간단한 샘플 코드입니다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# 경고 무시
warnings.filterwarnings("ignore")

# Data 경로 설정
DATA_DIR = "data"

In [ ]:
# 학습에 사용할 과거 data set 로드 (dataset.csv)
dataset = pd.read_csv(os.path.join(DATA_DIR, "dataset.csv"))

# problem set 로드 (problem.csv)
problemset = pd.read_csv(os.path.join(DATA_DIR, "problem.csv"))

In [ ]:
# 학습에 사용할 dataset 구성 확인
dataset 

### **<데이터 구성>**

* id
* Area (면적): 콩 영역 내부의 픽셀 개수
* Perimeter (둘레): 콩 윤곽선(경계)의 길이
* ConvexArea (볼록껍질 면적): 콩을 둘러싸는 가장 작은 볼록 다각형(convex hull)의 면적
* EquivDiamter (등가 지름): 콩과 동일한 면적을 가진 원의 지름
* MajorAxisLength (장축 길이): 콩을 관통하는 가장 긴 직선의 길이
* MinorAxisLength (단축 길이): 장축에 수직한 방향의 최대 길이
* AspectRation (종횡비): 단축/장축 비율
* Eccentricity (이심률): 콩을 타원으로 근사했을 때의 이심률
* Extent (충실도/외접비): 콩을 감싸는 최소 직사각형 대비 실제 콩의 면적 비율
* Solidity (견고도): 볼록껍질 면적(ConvexArea) 대비 면적(area) 비율 --> 콩이 얼마나 볼록한지를 나타냄
* Roundness (원형도): 완벽한 원이면 1, 윤곽이 길쭉하거나 울퉁불퉁할수록 값이 작아짐
* Compactness (조밀도): 같은 면적의 원의 지름이 실제 장축 길이에 비해 얼마나 되는지를 나타낸 비율 --> 원형일수록 1에 가깝고, 길쭉할수록 작아짐 (roundness 와 비슷한 개념이지만 둘레가 아닌 장축을 기준으로 평가)
* ShapeFactor1 : 면적당 장축 길이 (MajorAxisLength/Area)
* ShapeFactor2: 면적당 단축 길이 (MinorAxisLength/Area)
* ShapeFactor3: 장축을 지름으로 하는 원의 면적 대비 실제 면적의 비
* ShapeFactor4: 장축과 단축으로 정의되는 타원의 면적 대비 실제 면적의 비

In [ ]:
# problem set 확인: 400건의 문제 데이터셋 (품종을 예측해야 함)
problemset

## 데이터 전처리 및 모델 학습

In [ ]:
# data 와 problem 데이터들을 합쳐서 하나의 데이터로 만들어줍니다 --> 아래의 데이터 전처리 후 다시 분리할 예정
all_data = pd.concat([dataset, problemset], ignore_index=True)

# 결측치를 0으로 대체
all_data = all_data.fillna(0)

# Id 열을 제거
all_data = all_data.drop(["id"], axis=1)

In [ ]:
# 학습 데이터와 문제 데이터로 다시금 분리
train_data = all_data[: len(dataset)]
problem_data = all_data[len(dataset):]

In [ ]:
# 학습 데이터의 Class 열을 제외한 나머지 열을 X로 지정, Class 열을 Y로 지정
X = train_data.drop("Class", axis=1)
Y = train_data["Class"]

In [ ]:
# 모델 학습을 위해 학습 데이터를 80%의 학습 데이터(train)와 20%의 검증 데이터(test)로 나눔
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2)

In [ ]:
# 랜덤 포레스트 분류 모델을 사용하여 학습
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
model.fit(x_train, y_train)

In [ ]:
# train 데이터와 test 데이터에 대한 예측값을 구하고 분류 성능을 계산
from sklearn.metrics import accuracy_score, classification_report

train_pred = model.predict(x_train)
test_pred = model.predict(x_test)

print("Train Accuracy :", accuracy_score(y_train, train_pred))
print("Test Accuracy :", accuracy_score(y_test, test_pred))
print("\nClassification Report (Test):")
print(classification_report(y_test, test_pred))

Accuracy: 전체 샘플 중 정답을 맞춘 비율

## Problem set 문제에 대한 품종 예측 및 리더보드 결과 제출

- 아래 제출 프로세스가 느리다고 중지 후 다시 코드를 여러차례 재실행하는 경우 패널티가 발생할 수 있습니다. (제출 과정에서 제출 횟수 이슈 발생 가능: 하루 최대 200회 까지 가능)
- 제출에 성공할 경우, "제출에 성공하였습니다"의 메세지와 함께 제출 결과가 화면에 출력됩니다.
- 제출결과는 또한 [대회 페이지(리더보드 서버)](https://agtechresearch.pythonanywhere.com/competitions/globalbeans/)의 `리더보드` 와 `제출` 탭에서 확인할 수 있습니다.


In [ ]:
# 전처리 과정 중에 Class 가 0으로 채워져 있기 때문에, problem_data 에서 Class를 다시 제거
problem_data = problem_data.drop("Class", axis=1)

# 문제 데이터(problem data)에 대한 예측값을 구함
problem_pred = model.predict(problem_data)

In [ ]:
# 리더보드 서버 제출을 위한 파일 생성
submission = pd.read_csv(os.path.join(DATA_DIR, "submission.csv"))
submission["Class"] = problem_pred

# 예측 결과 화면에 출력 후 제출
display(submission)
competition.submit(project, username, password, submission)